# RoadGuard Qro — 03: YOLO11n Training on Colab / Kaggle
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlarconCastilloDiegoAndre/RoadGuard-Qro/blob/main/ml/03_train_yolo.ipynb)

Trains Ultralytics **YOLO11n** for 4-class road damage detection with transfer learning from COCO,
saving checkpoints directly to Google Drive.


In [ ]:
# Step 1: Verify GPU Environment & Mount Google Drive
import os
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

if 'google.colab' in str(get_ipython()):
    from google.colab import drive
    drive.mount('/content/drive')
    CHECKPOINT_DIR = "/content/drive/MyDrive/RoadGuard/runs"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
else:
    CHECKPOINT_DIR = "ml/runs"

!pip install -q ultralytics onnx onnxruntime albumentations


In [ ]:
# Step 2: Initialize YOLO11n Model
from ultralytics import YOLO

# Load pre-trained weights on COCO
model = YOLO('yolo11n.pt')
print("Model Architecture Summary:")
model.info()


In [ ]:
# Step 3: Run Model Training
# Dataset yaml path generated by notebook 01
DATASET_YAML = "ml/data/yolo_rdd/rdd2022.yaml"

results = model.train(
    data=DATASET_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    cos_lr=True,
    optimizer='AdamW',
    lr0=0.001,
    weight_decay=0.0005,
    # Domain specific data augmentations for vehicle dashcam footage
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    mosaic=1.0,
    project=CHECKPOINT_DIR,
    name='roadguard_yolo11n',
    exist_ok=True
)


In [ ]:
# Step 4: Validate on Test Split
metrics = model.val(data=DATASET_YAML, split='test')

print("\n=== TEST METRICS ===")
print(f"mAP@50:    {metrics.box.map50:.4f}")
print(f"mAP@50-95: {metrics.box.map:.4f}")

for i, cname in enumerate(model.names.values()):
    print(f"Class {cname:<18} Precision: {metrics.box.p[i]:.3f} | Recall: {metrics.box.r[i]:.3f} | mAP50: {metrics.box.ap50[i]:.3f}")


In [ ]:
# Step 5: Visualize Predictions on Test Images
import glob
import matplotlib.pyplot as plt
import cv2

test_images = glob.glob("ml/data/yolo_rdd/images/test/*.jpg")[:6]
results = model.predict(test_images, conf=0.25)

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, r in zip(axes.flatten(), results):
    im_bgr = r.plot()
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    ax.imshow(im_rgb)
    ax.axis("off")

plt.tight_layout()
plt.show()
